In [1]:
# load modules
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from scipy.special import gammaln
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os

In [4]:
# set consts
proj_dir = '/scratch/ti12/speech_247'
data_dir = f'{proj_dir}/anilu_comparison'
patient_list = glob(f'{data_dir}/Y*')
patient_list = [os.path.basename(item) for item in patient_list]

In [5]:
# get data by patient
gpt_embeddings = {}
for patient in patient_list:
    try:
        print("starting patient:", patient)
        embedding_path = f'{data_dir}/{patient}/embeddings/word_embeddings_gpt2.npy'
        gpt_embedding = np.load(embedding_path, mmap_mode='r')
        layer_36 = gpt_embedding[:, -1].copy()
        gpt_embeddings[patient] = np.array(layer_36)
    except Exception as e:
        print("couldn't load for patient:", e)

starting patient: YEY
starting patient: YEZ
starting patient: YFA
starting patient: YFC
starting patient: YFF
starting patient: YFG
starting patient: YFI
starting patient: YFK
starting patient: YFM
couldn't load for patient: [Errno 2] No such file or directory: '/scratch/ti12/speech_247/anilu_comparison/YFM/embeddings/word_embeddings_gpt2.npy'


In [6]:
# get spike counts by patient
spike_counts = {}

for patient in gpt_embeddings.keys():
    counts_path = f'{data_dir}/{patient}/spikes/word_spikes.npy'
    counts = np.load(counts_path)
    spike_counts[patient] = counts

In [9]:
# get spike durs by patient
spike_durs = {}
for patient in gpt_embeddings.keys():
    durs_path = f'{data_dir}/{patient}/words/word_info.csv'
    word_df = pd.read_csv(durs_path, index_col=0)
    durs = word_df['Duration'].values
    eps = 1e-6
    log_durs = np.log(np.maximum(durs, eps))
    spike_durs[patient] = log_durs

In [11]:
def nan_clean_XY(X, Y, durations=None):
    """
    1. Drop rows where *all* X values are NaN or *all* Y values are NaN.
    2. Keep everything else (including Y with some NaNs).
    """
    mask_keep = ~np.isnan(X).all(axis=1)
    mask_keep_y = ~np.isnan(Y).all(axis=1)
    if durations is not None:
        return X[mask_keep & mask_keep_y], Y[mask_keep & mask_keep_y], durations[mask_keep & mask_keep_y]
    return X[mask_keep & mask_keep_y], Y[mask_keep & mask_keep_y]

In [12]:
def impute_Y_all(Y):
    """
    Impute NaNs in Y using per-channel nanmean from training set only.
    Returns imputed copies of Y_train and Y_test.
    """
    Y_imp = Y.copy()
    # Compute per-channel mean ignoring NaNs
    channel_means = np.nanmean(Y_imp, axis=0)
    # Replace NaN means with 0 if an entire channel is NaN in train
    channel_means = np.where(np.isnan(channel_means), 0.0, channel_means)
    # get channel_means as int
    channel_means = np.round(channel_means).astype(int)
    # Fill NaNs
    inds = np.where(np.isnan(Y_imp))
    Y_imp[inds] = np.take(channel_means, inds[1])
    return Y_imp

In [13]:
# =============== Utilities ===============


def to_device(x, device):
    return torch.as_tensor(x, dtype=torch.float32, device=device)

def standardize_fit(X):
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0, ddof=0)
    sd[sd == 0] = 1.0
    return mu, sd

def standardize_apply(X, mu, sd):
    return (X - mu) / sd

def backtransform_coef(w_std_np, b_std_float, mu_np, sd_np):
    # w_raw = w_std / sd ;  intercept_raw = b_std - sum(mu * w_raw)
    w_raw = w_std_np / sd_np
    intercept_raw = b_std_float - float(np.dot(mu_np, w_raw))
    return w_raw, intercept_raw

def poisson_ll_numpy(y_true, mu_pred):
    mu_pred = np.clip(mu_pred, 1e-10, None)
    return float(np.sum(y_true * np.log(mu_pred) - mu_pred - gammaln(y_true + 1)))

def pseudo_r2(ll_model, ll_null):
    return 1.0 - (ll_model / ll_null)

In [14]:
# Hutchinson approximation of trace( B A^{-1} ), where:
#   B = X^T W X, A = B + alpha I, W = diag(mu_train)
def approx_edf_ridge_poisson_torch(Xs_t, mu_t, alpha, n_probe=64):
    # Xs_t: (n,d) standardized train tensor; mu_t: (n,)
    d = Xs_t.shape[1]
    XT_W = (Xs_t.T * mu_t)              # d x n
    B = XT_W @ Xs_t                     # d x d
    A = B + alpha * torch.eye(d, device=Xs_t.device)
    z = torch.randn(d, n_probe, device=Xs_t.device)
    v = torch.linalg.solve(A, z)        # A^{-1} z
    Bv = B @ v
    # trace(B A^{-1}) ≈ mean_i z_i^T (B A^{-1}) z_i
    tr_est = torch.sum(z * Bv, dim=0).mean()
    return float(tr_est.item())

In [15]:
import torch.nn as nn

# =============== Poisson Ridge model ===============

class PoissonRidgeTorch(nn.Module):
    def __init__(self, d, alpha=1.0):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(d))
        self.b = nn.Parameter(torch.zeros(()))
        self.alpha = float(alpha)

    def set_params(self, w0=None, b0=None):
        with torch.no_grad():
            if w0 is not None:
                self.w.copy_(w0)
            if b0 is not None:
                self.b.copy_(b0)

    def forward(self, X, offset=None):
        eta = X @ self.w + self.b
        if offset is not None:
            eta = eta + offset
        return torch.exp(eta)  # μ = exp(η) (log link)

    def loss(self, X, y, offset=None):
        mu = self.forward(X, offset)
        nll = torch.sum(mu - y * torch.log(mu.clamp_min(1e-10)))
        reg = 0.5 * self.alpha * torch.sum(self.w * self.w)
        return nll + reg

In [16]:
from torch.optim import LBFGS, Adam
# =============== Fitter (LBFGS or Adam+LBFGS) ===============

def fit_poisson_ridge_lbfgs(
    X_t, y_t, alpha, *,
    offset_t=None,
    init_w=None, init_b=None,
    max_iter=200, tol=1e-6,
    use_full_batch=True, batch_size=65536, adam_steps=None
):
    n, d = X_t.shape
    model = PoissonRidgeTorch(d, alpha=alpha).to(X_t.device)
    if init_w is not None or init_b is not None:
        model.set_params(init_w, init_b)

    if use_full_batch:
        optimizer = LBFGS(model.parameters(), lr=1.0, max_iter=max_iter,
                          tolerance_grad=tol, tolerance_change=tol,
                          history_size=10, line_search_fn='strong_wolfe')
        def closure():
            optimizer.zero_grad(set_to_none=True)
            loss = model.loss(X_t, y_t, offset_t)
            loss.backward()
            return loss
        optimizer.step(closure)
    else:
        # Mini-batch Adam warm-up (for memory-constrained cases), then full-batch LBFGS polish
        opt = Adam(model.parameters(), lr=1e-2)
        if adam_steps is None:
            adam_steps = min(2000, max(400, 4 * (n // batch_size + 1)))
        for _ in range(adam_steps):
            idx = torch.randint(0, n, (min(batch_size, n),), device=X_t.device)
            loss = model.loss(X_t[idx], y_t[idx], None if offset_t is None else offset_t[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        optimizer = LBFGS(model.parameters(), lr=1.0, max_iter=max_iter//2,
                          tolerance_grad=tol, tolerance_change=tol,
                          history_size=10, line_search_fn='strong_wolfe')
        def closure2():
            optimizer.zero_grad(set_to_none=True)
            loss = model.loss(X_t, y_t, offset_t)
            loss.backward()
            return loss
        optimizer.step(closure2)

    with torch.no_grad():
        w = model.w.detach().clone()
        b = model.b.detach().clone()
    return model, w, b


In [17]:
from sklearn.decomposition import PCA
def _prep_X_with_pca(
    X_tr_raw, X_te_raw,
    n_components=100,
):
    """
    Fit standardization + PCA + standardization-in-PCA-space on X_tr_raw only,
    then transform X_te_raw. Returns (Xtr_pca_std, Xte_pca_std, bundle)
    bundle contains everything needed for coef backtransform if desired.
    """
    # standardize in raw space
    mu_raw, sd_raw = standardize_fit(X_tr_raw)
    Xtr_std = standardize_apply(X_tr_raw, mu_raw, sd_raw)
    Xte_std = standardize_apply(X_te_raw, mu_raw, sd_raw)

    # PCA fit on training only
    pca = PCA(n_components=n_components)
    Xtr_pca = pca.fit_transform(Xtr_std)
    Xte_pca = pca.transform(Xte_std)

    # standardize in PCA space (fit on train only)
    mu_pca, sd_pca = standardize_fit(Xtr_pca)
    Xtr_pca_std = standardize_apply(Xtr_pca, mu_pca, sd_pca)
    Xte_pca_std = standardize_apply(Xte_pca, mu_pca, sd_pca)

    bundle = {
        "mu_raw": mu_raw, "sd_raw": sd_raw,
        "pca": pca,
        "mu_pca": mu_pca, "sd_pca": sd_pca,
    }
    return Xtr_pca_std, Xte_pca_std, bundle

In [18]:
def _tune_alpha_inner_cv_gpu(
    X_tr_raw, y_tr_np,
    off_tr_np,
    *,
    alphas,
    inner_splits=5,
    seed=42,
    device="cuda",
    n_pca_components=100,
    use_full_batch=True,
    batch_size=65536,
    max_iter=200,
    tol=1e-6,
):
    """
    Inner CV alpha tuning on provided TRAIN data only.
    Returns: best_alpha, alpha_ll_mean, alpha_ll_std, warmstart_wb
    where warmstart_wb is (w,b) from the last fit of an arbitrary fold (optional warmstart).
    """
    kf = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)
    splits = list(kf.split(X_tr_raw, y_tr_np))

    # Precompute fold tensors for speed; PCA/standardization are done per-fold (correct)
    fold_data = []
    for (tr_f, va_f) in splits:
        X_tr_f, X_va_f = X_tr_raw[tr_f], X_tr_raw[va_f]
        y_tr_f, y_va_f = y_tr_np[tr_f], y_tr_np[va_f]
        off_tr_f = off_va_f = None
        if off_tr_np is not None:
            off_tr_f, off_va_f = off_tr_np[tr_f], off_tr_np[va_f]

        Xtr_pca_std, Xva_pca_std, _ = _prep_X_with_pca(
            X_tr_f, X_va_f, n_components=n_pca_components
        )

        fold_data.append({
            "Xtr_t": to_device(Xtr_pca_std, device),
            "ytr_t": to_device(y_tr_f, device),
            "Xva_t": to_device(Xva_pca_std, device),
            "yva_np": y_va_f,  # keep numpy for LL
            "off_tr_t": to_device(off_tr_f, device) if off_tr_f is not None else None,
            "off_va_t": to_device(off_va_f, device) if off_va_f is not None else None,
        })

    alphas_desc = np.array(sorted(alphas, reverse=True), dtype=float)

    # warm-start cache per fold
    fold_cache = {i: (None, None) for i in range(len(fold_data))}
    alpha_to_scores = {a: [] for a in alphas_desc}

    for a in alphas_desc:
        for fidx, fd in enumerate(fold_data):
            w0, b0 = fold_cache[fidx]
            model, w, b = fit_poisson_ridge_lbfgs(
                fd["Xtr_t"], fd["ytr_t"], alpha=a,
                offset_t=fd["off_tr_t"],
                init_w=w0, init_b=b0,
                max_iter=max_iter, tol=tol,
                use_full_batch=use_full_batch, batch_size=batch_size
            )
            fold_cache[fidx] = (w, b)

            with torch.no_grad():
                mu_va = model(fd["Xva_t"], fd["off_va_t"]).clamp_min(1e-10).cpu().numpy()
            ll_va = poisson_ll_numpy(fd["yva_np"], mu_va)
            alpha_to_scores[a].append(ll_va)

    alpha_means = {a: float(np.mean(v)) for a, v in alpha_to_scores.items()}
    best_alpha = max(alpha_means, key=alpha_means.get)
    alpha_ll_mean = alpha_means[best_alpha]
    alpha_ll_std = float(np.std(alpha_to_scores[best_alpha]))

    # any fold warmstart for outer final fit
    any_wb = next(iter(fold_cache.values()))
    return float(best_alpha), float(alpha_ll_mean), float(alpha_ll_std), any_wb

In [19]:
def run_poisson_ridge_gpu_nested_cv(
    X_full, Y, neuron_idx, *,
    seed=42,
    outer_splits=5,
    inner_splits=5,
    alphas=np.logspace(-3, 3, 30),
    n_shuffles=0,                 # set >0 if you want permutation baselines per fold (expensive)
    device="cuda",
    use_full_batch=True,
    batch_size=65536,
    offset_full=None,
    n_probe_edf=64,
    n_pca_components=100,
    all_data_run=False
):
    """
    Nested CV:
      outer K-fold -> unbiased test metrics
      inner K-fold -> tune alpha inside each outer-train

    Returns:
      model_full (fit on ALL data with alpha chosen by inner-CV on ALL data; optional convenience)
      results dict with:
        fold_metrics: list of per-fold dicts
        summary: aggregated mean/std across folds for key metrics
    """
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)

    y_all = Y[:, neuron_idx].astype(np.float32, copy=False)
    if np.std(y_all) == 0 or np.all(y_all == 0) or not np.all(np.isfinite(y_all)):
        return None, {"neuron": int(neuron_idx), "fold_metrics": [], "summary": None}

    kf_outer = KFold(n_splits=outer_splits, shuffle=True, random_state=seed)

    fold_metrics = []
    for fold_i, (tr_idx, te_idx) in enumerate(kf_outer.split(X_full, y_all)):
        print("starting fold:", fold_i)
        X_tr_raw, X_te_raw = X_full[tr_idx], X_full[te_idx]
        y_tr_np, y_te_np   = y_all[tr_idx], y_all[te_idx]
        off_tr_np = off_te_np = None
        if offset_full is not None:
            off_tr_np, off_te_np = offset_full[tr_idx], offset_full[te_idx]

        # ---- inner CV tune on outer-train only ----
        print("picking best alpha (warm starts)")
        best_alpha, alpha_ll_mean, alpha_ll_std, (init_w, init_b) = _tune_alpha_inner_cv_gpu(
            X_tr_raw, y_tr_np, off_tr_np,
            alphas=alphas,
            inner_splits=inner_splits,
            seed=seed + 1000 * fold_i,   # vary deterministically by fold
            device=device,
            n_pca_components=n_pca_components,
            use_full_batch=use_full_batch,
            batch_size=batch_size,
        )

        # ---- fit on full outer-train with best alpha ----
        print("fitting final model")
        Xtr_pca_std, Xte_pca_std, bundle = _prep_X_with_pca(
            X_tr_raw, X_te_raw, n_components=n_pca_components
        )
        X_tr_t = to_device(Xtr_pca_std, device)
        y_tr_t = to_device(y_tr_np, device)
        X_te_t = to_device(Xte_pca_std, device)
        y_te_t = to_device(y_te_np, device)
        off_tr_t = to_device(off_tr_np, device) if off_tr_np is not None else None
        off_te_t = to_device(off_te_np, device) if off_te_np is not None else None

        model, w_std_t, b_std_t = fit_poisson_ridge_lbfgs(
            X_tr_t, y_tr_t, alpha=best_alpha,
            offset_t=off_tr_t,
            init_w=init_w, init_b=init_b,
            max_iter=300, tol=1e-6,
            use_full_batch=use_full_batch, batch_size=batch_size
        )

        with torch.no_grad():
            mu_te = model(X_te_t, off_te_t).clamp_min(1e-10).cpu().numpy()

        ll_real = poisson_ll_numpy(y_te_np, mu_te)

        # null model (uses train mean rate)
        if off_tr_np is not None:
            avg_rate = y_tr_np.sum() / np.exp(off_tr_np).sum()
            mu_null = avg_rate * np.exp(off_te_np)
        else:
            avg_rate = y_tr_np.mean()
            mu_null = np.full_like(y_te_np, avg_rate, dtype=np.float32)
        ll_null = poisson_ll_numpy(y_te_np, mu_null)

        pr2 = float(pseudo_r2(ll_real, ll_null))
        pear = float(pearsonr(y_te_np, mu_te)[0]) if np.std(mu_te) > 0 else np.nan
        spear = float(spearmanr(y_te_np, mu_te)[0]) if np.std(mu_te) > 0 else np.nan

        # EDF/AIC/BIC: compute EDF on TRAIN; AIC/BIC usually use ll on TEST here (your prior behavior)
        with torch.no_grad():
            mu_tr_fit = model(X_tr_t, off_tr_t)
        edf = float(approx_edf_ridge_poisson_torch(X_tr_t, mu_tr_fit, best_alpha, n_probe=n_probe_edf))
        aic = float(2 * edf - 2 * ll_real)
        bic = float(np.log(len(y_te_np)) * edf - 2 * ll_real)

        # optional coef backtransform (your backtransform_coef currently assumes only PCA-space standardization;
        # if you also want to go all the way back through PCA + raw std you’ll need a different function.
        w_std_np = w_std_t.detach().cpu().numpy()
        b_std = float(b_std_t.detach().cpu().item())
        # This matches your existing behavior (backtransform from PCA-standardized -> PCA coords)
        w_pca, intercept_pca = backtransform_coef(w_std_np, b_std, bundle["mu_pca"], bundle["sd_pca"])

        # ---- optional permutation baseline per fold (expensive) ----
        ll_shufs = None
        p_val_ll_xshuf = None
        ll_xshuf_mean = None
        ll_diff = None
        if n_shuffles and n_shuffles > 0:
            print("doing permutation tests")
            ll_shufs = []
            for _ in range(n_shuffles):
                perm = rng.permutation(len(X_full))
                X_shuf_tr_raw = X_full[perm][tr_idx]
                X_shuf_te_raw = X_full[perm][te_idx]

                # IMPORTANT: refit preprocess on shuffled TRAIN only (no leakage)
                Xs_tr, Xs_te, _ = _prep_X_with_pca(
                    X_shuf_tr_raw, X_shuf_te_raw, n_components=n_pca_components
                )
                Xs_tr_t = to_device(Xs_tr, device)
                Xs_te_t = to_device(Xs_te, device)

                m_shuf, _, _ = fit_poisson_ridge_lbfgs(
                    Xs_tr_t, y_tr_t, alpha=best_alpha,
                    offset_t=off_tr_t,
                    max_iter=200, tol=1e-6,
                    use_full_batch=use_full_batch, batch_size=batch_size
                )
                with torch.no_grad():
                    mu_s_te = m_shuf(Xs_te_t, off_te_t).clamp_min(1e-10).cpu().numpy()
                ll_shufs.append(poisson_ll_numpy(y_te_np, mu_s_te))

            ll_xshuf_mean = float(np.mean(ll_shufs))
            ll_diff = float(ll_real - ll_xshuf_mean)
            p_val_ll_xshuf = float((np.sum(np.array(ll_shufs) >= ll_real) + 1) / (len(ll_shufs) + 1))

        fold_metrics.append({
            "state_dict": model.state_dict(), 
            "fold": fold_i,
            "best_alpha": best_alpha,
            "alpha_ll_mean": alpha_ll_mean,
            "alpha_ll_std": alpha_ll_std,
            "ll_real": float(ll_real),
            "ll_null": float(ll_null),
            "pseudo_r2": pr2,
            "pearson_corr": pear,
            "spearman_corr": spear,
            "edf": edf,
            "aic": aic,
            "bic": bic,
            "ll_shufs": np.array(ll_shufs, dtype=float),
            "ll_xshuf_mean": ll_xshuf_mean,
            "ll_diff": ll_diff,
            "p_val_ll_xshuf": p_val_ll_xshuf,
            "coef_pca_space": w_pca.astype(float).tolist(),
            # you can store intercept_pca too if useful
        })

    # ---- aggregate summary ----
    def _agg(key):
        vals = np.array([fm[key] for fm in fold_metrics], dtype=float)
        return {"mean": float(np.nanmean(vals)), "std": float(np.nanstd(vals))}

    p_val_ll_xshuf = None
    ll_xshuf_mean = None
    if n_shuffles and n_shuffles > 0:
        # observed aggregated statistic
        d_obs = np.array([fm["ll_real"] for fm in fold_metrics], dtype=float)
        T_obs = float(np.mean(d_obs))
    
        # build permutation aggregated statistics, aligned by permutation index
        K = len(fold_metrics)
        B = n_shuffles
    
        # shape: (K, B)
        ll_shufs_mat = np.vstack([fm["ll_shufs"] for fm in fold_metrics])
    
        T_perm = np.mean(ll_shufs_mat, axis=0)   # shape (B,)
    
        # one-sided p-value: how often perm >= obs
        p_val_ll_xshuf = float((np.sum(T_perm >= T_obs) + 1) / (B + 1))
        ll_xshuf_mean = np.mean(np.concatenate([fm["ll_shufs"] for fm in fold_metrics]))

    summary = {
        "neuron": int(neuron_idx),
        "outer_splits": int(outer_splits),
        "inner_splits": int(inner_splits),
        "best_alpha": _agg("best_alpha"),
        "ll_real": _agg("ll_real"),
        "pseudo_r2": _agg("pseudo_r2"),
        "pearson_corr": _agg("pearson_corr"),
        "spearman_corr": _agg("spearman_corr"),
        "p_val_ll_xshuf": p_val_ll_xshuf,
        "ll_xshuf_mean": ll_xshuf_mean, 
        "edf": _agg("edf"),
        "aic": _agg("aic"),
        "bic": _agg("bic"),
    }

    # ---- optional convenience fit on ALL data using inner-CV on ALL data ----
    if all_data_run:
        # This is *not* used for evaluation; it’s just to produce a single final model if you want to deploy/interpret.
        best_alpha_full, _, _, (init_w, init_b) = _tune_alpha_inner_cv_gpu(
            X_full, y_all, offset_full,
            alphas=alphas,
            inner_splits=inner_splits,
            seed=seed,
            device=device,
            n_pca_components=n_pca_components,
            use_full_batch=use_full_batch,
            batch_size=batch_size,
        )
        X_all_pca_std, _, bundle_all = _prep_X_with_pca(X_full, X_full, n_components=n_pca_components)
        X_all_t = to_device(X_all_pca_std, device)
        y_all_t = to_device(y_all, device)
        off_all_t = to_device(offset_full, device) if offset_full is not None else None
    
        model_full, w_full, b_full = fit_poisson_ridge_lbfgs(
            X_all_t, y_all_t, alpha=best_alpha_full,
            offset_t=off_all_t,
            init_w=init_w, init_b=init_b,
            max_iter=400, tol=1e-6,
            use_full_batch=use_full_batch, batch_size=batch_size
        )
            
        return model_full, {
            "neuron": int(neuron_idx),
            "fold_metrics": fold_metrics,
            "summary": summary,
            "best_alpha_full": float(best_alpha_full),
            "bundle_full": bundle_all,  # PCA/std params if you want to transform new data consistently
        }
    else:
        return None, {
            "neuron": int(neuron_idx),
            "fold_metrics": fold_metrics,
            "summary": summary,
            "best_alpha_full": None,
            "bundle_full": None,  # PCA/std params if you want to transform new data consistently
        }


In [111]:
def parallel_runner(X, Y, i, durations, n_shuffles=20):
    try:
        return run_poisson_ridge_gpu_nested_cv(X, Y, i, offset_full=durations, n_shuffles=n_shuffles)
    except Exception as e:
        return None

In [ ]:
# now get poisson ridge results for each neuron
from joblib import Parallel, delayed
import dill as pickle
import torch
poisson_dfs = {}
poisson_models_all = {}
for patient in patient_list:
    # skip patients for which we already have results
    res_path = f'{data_dir}/{patient}/results/word_encoding_results_cv.pkl'
    if os.path.exists(res_path):
        print("already have results for", patient)
        continue
    print("starting patient", patient)
    poisson_models = {}
    poisson_res = []
    embeddings = gpt_embeddings[patient]
    durations = spike_durs[patient]
    
    # get results for offset 0
    spikes = spike_counts[patient]
    _, n_neuron = spikes.shape
    X, Y, dur_clean = nan_clean_XY(embeddings, spikes, durations=durations)
    Y = impute_Y_all(Y)
    results = Parallel(n_jobs=3, verbose=100, return_as="generator", backend="threading")(
            (delayed(parallel_runner)(X, Y, i, durations, n_shuffles=50) for i in range(n_neuron))
        )
    # sequential loop for debugging
    # results = []
    # for i in range(n_neuron):
    #     res = parallel_runner(X, Y, i, dur_clean, n_shuffles=20)
    #     results.append(res)
        
    for res in results:
        if res is not None:
            df_row = {}
            res_info = res[1]
            neuron_idx = res_info["neuron"]
            df_row["neuron_idx"] = neuron_idx
            df_row["best_alpha_whole"] = np.nan
            df_row["fold"] = np.nan
            df_row["outer_splits"] = res_info["summary"]["outer_splits"]
            df_row["inner"] = res_info["summary"]["inner_splits"]
            df_row["summary"] = False
            for fm in res_info["fold_metrics"]:
                f_row = df_row.copy()
                fold_id = fm["fold"]
                poisson_models[f"neur_{neuron_idx}_k_{fold_id}"] = fm["state_dict"]
                del fm["state_dict"]
                f_row = {**f_row, **fm}
                poisson_res.append(f_row)
            df_row["summary"] = True
            df_row = {**df_row, **res_info["summary"]}
            df_row["ll_shufs"] = np.nan
            poisson_res.append(df_row)
            

    # save to dataframe
    poisson_df = pd.DataFrame(poisson_res)
    # save results
    with open(f'{data_dir}/{patient}/results/word_encoding_results_cv.pkl', 'wb') as f:
        pickle.dump(poisson_df, f)
    torch.save(poisson_models, f'{data_dir}/{patient}/results/word_encoding_results_cv.tar')
    poisson_dfs[patient] = poisson_df
    poisson_models_all[patient] = poisson_models
    

starting patient YEY
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
starting fold: 0
starting fold: 0
starting fold: 0
picking best alpha (warm starts)
picking best alpha (warm starts)
picking best alpha (warm starts)
fitting final model
fitting final model
fitting final model
doing permutation tests
doing permutation tests
doing permutation tests
starting fold: 1
picking best alpha (warm starts)
starting fold: 1
picking best alpha (warm starts)
starting fold: 1
picking best alpha (warm starts)
fitting final model
fitting final model
fitting final model
doing permutation tests
doing permutation tests
doing permutation tests
starting fold: 2
picking best alpha (warm starts)
starting fold: 2
picking best alpha (warm starts)
starting fold: 2
picking best alpha (warm starts)
fitting final model
fitting final model
doing permutation tests
fitting final model
doing permutation tests
doing permutation tests
starting fold: 3
picking best alpha (warm starts)
sta

In [24]:
# load dfs if needed
poisson_dfs = {}
for patient in patient_list:
    poisson_df = pd.read_csv(f'{data_dir}/{patient}/all_convo_recording/word_encoding_results_w2v.csv', index_col=0)
    poisson_dfs[patient] = poisson_df

In [21]:
# now load micro dfs so you can align our results to each region
micro_dfs = {}
for patient in patient_list:
    micro_df = pd.read_csv(f'{proj_dir}/vad_out/{patient}/{patient}_micro_info.csv', index_col=0)
    micro_dfs[patient] = micro_df

In [31]:
# now create micro aligned result of df
res_dfs = {}
for patient in patient_list:
    poisson_df = poisson_dfs[patient]
    spikes = spike_counts[patient]
    embeddings = bert_embeddings[patient]
    res_df = micro_dfs[patient].copy()
    # set columns to inherit from results
    res_df['ll_diff'] = None
    res_df['p_val_ll_xshuf'] = None
    res_df['ll_diff'] = poisson_df['ll_diff'] 
    res_df['p_val_ll_xshuf'] = poisson_df['p_val_ll_xshuf'] 
    # res_df.to_csv(f'{data_dir}/{patient}/all_convo_recording/word_encoding_results_concat.csv')
    res_dfs[patient] = res_df